DATASET GENERATOR

In [1]:
import pandas as pd

In [2]:
# 1. Caricamento dei file
df_metadati = pd.read_csv('audio_samples_filtered_metadata.csv')
df_features = pd.read_csv('audio_samples_filtered_features.csv')
df_portate = pd.read_csv('ValoriPortata.csv', sep=';')
CSV_OUTPUT_PATH = 'audio_dataset.csv'

In [3]:
# 2. Conversione dei timestamp in oggetti datetime
# Assicurati che il formato sia corretto per entrambi i dataframe
df_metadati['timestamp'] = pd.to_datetime(df_metadati['timestamp'])
#print(df_metadati["timestamp"].head(1))
df_metadati['timestamp'] = df_metadati['timestamp'].dt.tz_convert('Europe/Rome')
#print(df_metadati["timestamp"].head(1))
df_portate['timestamp'] = pd.to_datetime(df_portate['timestamp'])
#print(df_portate["timestamp"].head(1))
df_portate['timestamp'] = df_portate['timestamp'].dt.tz_localize('Europe/Rome',nonexistent='shift_forward', ambiguous='infer')
#print(df_portate["timestamp"].head(1))

cols_to_drop = [c for c in df_metadati.columns if c != 'filename']

# 3. Unione di metadati e feature (basata sul filename)
df_audio = pd.merge(df_metadati, df_features, on='filename')

# 4. Ordinamento per timestamp
# Fondamentale per far funzionare merge_asof
df_audio = df_audio.sort_values('timestamp')
df_portate = df_portate.sort_values('timestamp')

# 5. Merge temporale "approssimativo"
# Questo comando cerca per ogni riga di df_audio il valore di df_portate 
# con il timestamp più vicino.
df_final = pd.merge_asof(
    df_audio, 
    df_portate, 
    on='timestamp', 
    direction='nearest'
)

df_final = df_final.drop(columns=cols_to_drop)
df_final.set_index('filename', inplace=True)

In [4]:
# Visualizzazione del risultato
print(df_final.head())

                                ae_mean    ae_std    ae_min    ae_max  \
filename                                                                
audio_2026-02-23T09-26-59.wav  0.000615  0.002290  0.000000  0.023438   
audio_2026-02-23T09-28-11.wav  0.000615  0.002133  0.000000  0.020569   
audio_2026-02-23T10-37-03.wav  0.000095  0.001485 -0.003235  0.013702   
audio_2026-02-23T10-38-15.wav  0.000396  0.001803 -0.003876  0.014862   
audio_2026-02-23T10-39-26.wav  0.000200  0.001581 -0.003998  0.013458   

                                  ae_q1  ae_median     ae_q3    ae_iqr  \
filename                                                                 
audio_2026-02-23T09-26-59.wav  0.000061   0.000122  0.000427  0.000366   
audio_2026-02-23T09-28-11.wav  0.000061   0.000153  0.000427  0.000366   
audio_2026-02-23T10-37-03.wav -0.000092   0.000000  0.000061  0.000153   
audio_2026-02-23T10-38-15.wav -0.000122   0.000000  0.000153  0.000275   
audio_2026-02-23T10-39-26.wav -0.000061   0.

In [5]:
# Salvataggio
df_final.to_csv(CSV_OUTPUT_PATH, index=True)